# Metaflora Incubus v1: private head-to-head

This notebook starts from a fresh Kaggle session. It restores the candidate, incumbent and pinned runtime from the private checkpoint branch, builds the runtime from the pinned source revision only when restoration is unavailable, downloads the promotion-required baselines, verifies every byte, runs the paired benchmark, and stores private decision evidence. Interrupted downloads reuse the Hugging Face cache. Nothing is published.

In [ ]:
import hashlib
import json
import os
import re
import shutil
import subprocess
import sys
from pathlib import Path

WORK_ROOT = Path("/kaggle/working")
REPO = WORK_ROOT / "metaflora-incubus-head-to-head"
MODEL_DIR = WORK_ROOT / "head-to-head-models"
OUTPUT_DIR = WORK_ROOT / "head-to-head-output"
MANIFEST_PATH = WORK_ROOT / "head-to-head-v1.json"
if not WORK_ROOT.is_dir():
    raise RuntimeError("This notebook must run in a Kaggle session")
code_revision = "7ec9bcd46001b0ecd8d15e83203835f06dca59ea"
if re.fullmatch(r"[0-9a-f]{40}", code_revision) is None:
    raise RuntimeError("INCUBUS_CODE_REVISION must be an exact lowercase commit SHA")
shutil.rmtree(REPO, ignore_errors=True)
subprocess.run(["git", "init", str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "remote", "add", "origin", "https://github.com/metaflora-app/metaflora-incubus.git"], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", code_revision], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", "FETCH_HEAD"], check=True)
checked_out = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip().lower()
if checked_out != code_revision:
    raise RuntimeError("trusted code revision verification failed")
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--require-hashes", "-r", str(REPO / "requirements/h2h-linux.lock")], check=True)
sys.path.insert(0, str(REPO / "src"))
from huggingface_hub import HfApi, hf_hub_download
from metaflora_incubus.cloud_bootstrap import decrypt_cloud_bootstrap, install_cloud_bootstrap

encoded_bootstrap = Path("/kaggle/input/incubus-private-runtime-bootstrap/bootstrap-key.txt").read_text(encoding="utf-8").strip()
if not encoded_bootstrap.strip():
    raise RuntimeError("INCUBUS_BOOTSTRAP is required for private checkpoint access")
bootstrap_payload = decrypt_cloud_bootstrap((REPO / "configs/cloud/bootstrap-v1.enc").read_bytes(), encoded_bootstrap)
install_cloud_bootstrap(bootstrap_payload, home=Path.home(), environment=os.environ)
checkpoint_repo = os.environ.get("INCUBUS_CHECKPOINT_LOCATION", "metaflora/incubus-checkpoints")
checkpoint_branch = os.environ.get("INCUBUS_CHECKPOINT_BRANCH", "incubus-training-v1")
checkpoint_token = os.environ.get("HF_TOKEN") or None
checkpoint_head = HfApi(token=checkpoint_token).model_info(repo_id=checkpoint_repo, revision=checkpoint_branch).sha
if not isinstance(checkpoint_head, str) or re.fullmatch(r"[0-9a-f]{40}", checkpoint_head) is None:
    raise RuntimeError("private checkpoint revision could not be frozen")
candidate_pointer_name = "runs/incubus-v1-refine-001/exports/candidate-upload-receipt.json"
incumbent_model_name = "runs/incubus-v1-run/artifacts/metaflora-incubus-v1.gguf"
incumbent_receipt_name = "runs/incubus-v1-run/artifacts/artifact-metadata.json"
private_cache = MODEL_DIR / "private-checkpoint-cache"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
def download_private(name, revision):
    return Path(hf_hub_download(repo_id=checkpoint_repo, filename=name, revision=revision, cache_dir=private_cache, token=checkpoint_token, force_download=False))

candidate_pointer_path = Path(hf_hub_download(repo_id=checkpoint_repo, filename=candidate_pointer_name, revision=checkpoint_head, cache_dir=private_cache, token=checkpoint_token, force_download=False))
candidate_upload_receipt = json.loads(candidate_pointer_path.read_text(encoding="utf-8"))
if not isinstance(candidate_upload_receipt, dict):
    raise RuntimeError("candidate upload receipt must be a JSON object")
candidate_artifact_revision = candidate_upload_receipt["artifact_revision"]
candidate_artifact_path = candidate_upload_receipt["artifact_path"]
candidate_remote_prefix = candidate_upload_receipt["remote_prefix"]
candidate_artifact_sha256 = candidate_upload_receipt["artifact_sha256"]
candidate_artifact_size_bytes = candidate_upload_receipt["artifact_size_bytes"]
expected_remote_prefix = f"runs/incubus-v1-refine-001/exports/q5-k-m/{candidate_artifact_sha256}"
valid_candidate_pointer = (
    candidate_upload_receipt.get("schema_version") == 1
    and candidate_upload_receipt.get("run_id") == "incubus-v1-refine-001"
    and candidate_upload_receipt.get("repo_id") == checkpoint_repo
    and candidate_upload_receipt.get("branch") == checkpoint_branch
    and candidate_upload_receipt.get("gguf_quantization") == "Q5_K_M"
    and candidate_upload_receipt.get("release_ready") is False
    and candidate_upload_receipt.get("required_next_step") == "run_parity_and_release_gates"
    and isinstance(candidate_artifact_revision, str)
    and re.fullmatch(r"[0-9a-f]{40}", candidate_artifact_revision) is not None
    and isinstance(candidate_artifact_sha256, str)
    and re.fullmatch(r"[0-9a-f]{64}", candidate_artifact_sha256) is not None
    and isinstance(candidate_artifact_size_bytes, int)
    and not isinstance(candidate_artifact_size_bytes, bool)
    and 5 * 1024**3 // 2 <= candidate_artifact_size_bytes <= 5 * 1024**3
    and candidate_remote_prefix == expected_remote_prefix
    and candidate_artifact_path == f"{candidate_remote_prefix}/metaflora-incubus-v1.gguf"
)
if not valid_candidate_pointer:
    raise RuntimeError("candidate upload receipt failed its contract")
candidate_receipt_name = f"{candidate_remote_prefix}/candidate-export.json"
CANDIDATE_GGUF = download_private(candidate_artifact_path, revision=candidate_artifact_revision)
candidate_receipt = json.loads(download_private(candidate_receipt_name, revision=candidate_artifact_revision).read_text(encoding="utf-8"))
INCUMBENT_GGUF = download_private(incumbent_model_name, revision=checkpoint_head)
incumbent_receipt = json.loads(download_private(incumbent_receipt_name, revision=checkpoint_head).read_text(encoding="utf-8"))

llama_cpp = WORK_ROOT / "llama.cpp-head-to-head"
LLAMA_SERVER = llama_cpp / "build/bin/llama-server"
LLAMA_SERVER_BUILD_LOG = OUTPUT_DIR / "llama-server-build.log"
llama_cpp_revision = "bf2c86ddc0685f580595954056c2e77ebabfab4f"
if re.fullmatch(r"[0-9a-f]{40}", llama_cpp_revision) is None:
    raise RuntimeError("llama.cpp revision is not pinned")

def run_build_step(label, command):
    completed = subprocess.run(command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    with LLAMA_SERVER_BUILD_LOG.open("a", encoding="utf-8") as stream:
        stream.write(f"$ {subprocess.list2cmdline([str(item) for item in command])}\n")
        stream.write(completed.stdout or "")
        stream.write("\n")
    return completed

def raise_build_error(label, completed, summary):
    tail = (completed.stdout or "").strip().splitlines()[-20:]
    detail = "\n".join(tail) or "no compiler output captured"
    raise RuntimeError(f"{summary} during {label}; log: {LLAMA_SERVER_BUILD_LOG}\n{detail}")

shutil.rmtree(llama_cpp, ignore_errors=True)
LLAMA_SERVER_BUILD_LOG.unlink(missing_ok=True)
for label, command in (
    ("fetch", ["git", "init", str(llama_cpp)]),
    ("add-remote", ["git", "-C", str(llama_cpp), "remote", "add", "origin", "https://github.com/ggerganov/llama.cpp.git"]),
    ("fetch", ["git", "-C", str(llama_cpp), "fetch", "--depth", "1", "origin", llama_cpp_revision]),
    ("checkout", ["git", "-C", str(llama_cpp), "checkout", "--detach", "FETCH_HEAD"]),
):
    completed = run_build_step(label, command)
    if completed.returncode != 0:
        raise_build_error(label, completed, "llama-server build failed")
checked_out_llama_revision = subprocess.check_output(["git", "-C", str(llama_cpp), "rev-parse", "HEAD"], text=True).strip().lower()
if checked_out_llama_revision != llama_cpp_revision:
    raise RuntimeError("llama.cpp checkout verification failed")
cuda_configure = run_build_step("cuda_configure", ["cmake", "-S", str(llama_cpp), "-B", str(llama_cpp / "build"), "-DGGML_CUDA=ON", "-DLLAMA_CURL=OFF", "-DCMAKE_BUILD_TYPE=Release"])
cuda_compile = None
if cuda_configure.returncode == 0:
    cuda_compile = run_build_step("cuda_compile", ["cmake", "--build", str(llama_cpp / "build"), "--config", "Release", "-j", "2", "--target", "llama-server"])
server_backend = "cuda"
if cuda_configure.returncode != 0 or cuda_compile is None or cuda_compile.returncode != 0:
    server_backend = "cpu_fallback"
    shutil.rmtree(llama_cpp / "build", ignore_errors=True)
    cpu_configure = run_build_step("cpu_fallback_configure", ["cmake", "-S", str(llama_cpp), "-B", str(llama_cpp / "build"), "-DGGML_CUDA=OFF", "-DLLAMA_CURL=OFF", "-DCMAKE_BUILD_TYPE=Release"])
    if cpu_configure.returncode != 0:
        raise_build_error("cpu_fallback_configure", cpu_configure, "CPU fallback build failed")
    cpu_compile = run_build_step("cpu_fallback_compile", ["cmake", "--build", str(llama_cpp / "build"), "--config", "Release", "-j", "2", "--target", "llama-server"])
    if cpu_compile.returncode != 0:
        raise_build_error("cpu_fallback_compile", cpu_compile, "CPU fallback build failed")
if not LLAMA_SERVER.is_file():
    raise FileNotFoundError(f"llama-server build did not produce {LLAMA_SERVER}; inspect {LLAMA_SERVER_BUILD_LOG}")
LLAMA_SERVER.chmod(0o700)

CATALOG_PATH = REPO / "benchmarks/head-to-head-v1-baselines.json"
catalog = json.loads(CATALOG_PATH.read_text(encoding="utf-8"))
print({"repo": str(REPO), "candidate": str(CANDIDATE_GGUF), "incumbent": str(INCUMBENT_GGUF), "llama_cpp_revision": checked_out_llama_revision, "server_backend": server_backend, "server_build_log": str(LLAMA_SERVER_BUILD_LOG)})


In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def verify_gguf(path, expected_size=None, expected_sha256=None):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    with path.open("rb") as stream:
        if stream.read(4) != b"GGUF":
            raise RuntimeError(f"Invalid GGUF magic: {path}")
    size = path.stat().st_size
    if expected_size is not None and size != expected_size:
        raise RuntimeError(f"GGUF size mismatch: {path}")
    digest = sha256_file(path)
    if expected_sha256 is not None and digest != expected_sha256:
        raise RuntimeError(f"GGUF SHA-256 mismatch: {path}")
    return {"path": str(path), "size": size, "sha256": digest}

def atomic_json(path, document):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(document, ensure_ascii=False, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    temporary.replace(path)

candidate_artifact = candidate_receipt.get("artifact", {})
valid_candidate_export = (
    candidate_receipt.get("schema_version") == 1
    and candidate_receipt.get("run_id") == "incubus-v1-refine-001"
    and candidate_receipt.get("candidate_state") == "quantized_candidate"
    and candidate_receipt.get("gguf_quantization") == "Q5_K_M"
    and candidate_receipt.get("release_ready") is False
    and candidate_receipt.get("required_next_step") == "run_parity_and_release_gates"
    and candidate_artifact.get("path") == "metaflora-incubus-v1.gguf"
    and candidate_receipt["artifact"]["sha256"] == candidate_upload_receipt["artifact_sha256"]
    and candidate_artifact.get("size_bytes") == candidate_upload_receipt["artifact_size_bytes"]
)
if not valid_candidate_export:
    raise RuntimeError("candidate export receipt does not match its stable pointer")
candidate_verification = verify_gguf(CANDIDATE_GGUF, candidate_upload_receipt["artifact_size_bytes"], candidate_upload_receipt["artifact_sha256"])
incumbent_verification = verify_gguf(INCUMBENT_GGUF, incumbent_receipt.get("artifact_size_bytes"), incumbent_receipt["artifact_sha256"])
if not os.access(LLAMA_SERVER, os.X_OK):
    raise RuntimeError("llama-server is not executable")
required_pins = [pin for pin in catalog["baselines"] if pin["promotion_required"]]
if not required_pins:
    raise RuntimeError("Baseline catalog has no promotion-required artifacts")
verified_dir = MODEL_DIR / ".verified"
baseline_paths = {}
for pin in required_pins:
    destination = MODEL_DIR / pin["artifact_filename"]
    try:
        verification = verify_gguf(destination, pin["artifact_size_bytes"], pin["artifact_sha256"])
    except (FileNotFoundError, RuntimeError):
        destination.unlink(missing_ok=True)
        downloaded = Path(hf_hub_download(
            repo_id=pin["artifact_repo_id"],
            filename=pin["artifact_filename"],
            revision=pin["artifact_revision"],
            local_dir=MODEL_DIR,
            token=os.environ.get("HF_TOKEN") or None,
            force_download=False,
        ))
        verification = verify_gguf(downloaded, pin["artifact_size_bytes"], pin["artifact_sha256"])
        destination = downloaded
    marker = verified_dir / f"{pin['id']}.verified.json"
    atomic_json(marker, {"artifact_revision": pin["artifact_revision"], "artifact_sha256": verification["sha256"], "artifact_size_bytes": verification["size"], "path": str(destination)})
    baseline_paths[pin["id"]] = destination
print({"verified_required_baselines": sorted(baseline_paths)})


In [ ]:
models = [
    {"id": "candidate-v1", "role": "candidate", "path": str(CANDIDATE_GGUF), "sha256": candidate_verification["sha256"]},
    {"id": "incumbent-v1", "role": "incumbent", "path": str(INCUMBENT_GGUF), "sha256": incumbent_verification["sha256"]},
]
for pin in required_pins:
    models.append({"id": pin["id"], "role": "same_size", "path": str(baseline_paths[pin["id"]]), "sha256": pin["artifact_sha256"]})
runner_revision = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
launch_settings = catalog["launch_settings"]
execution_manifest = {
    "server_binary": str(LLAMA_SERVER),
    "server_sha256": sha256_file(LLAMA_SERVER),
    "cases_path": str(REPO / launch_settings["case_bank"]),
    "output_dir": str(OUTPUT_DIR),
    "runner_code_revision": runner_revision,
    "seed": launch_settings["seed"],
    "port": 18100,
    "gpu_layers": launch_settings["gpu_layers"] if server_backend == "cuda" else 0,
    "health_timeout_seconds": 120,
    "request_timeout_seconds": 120,
    "models": models,
}
atomic_json(MANIFEST_PATH, execution_manifest)
manifest_sha256 = sha256_file(MANIFEST_PATH)
print({"manifest": str(MANIFEST_PATH), "sha256": manifest_sha256, "models": [model["id"] for model in models]})


In [ ]:
REPORT_PATH = OUTPUT_DIR / "head-to-head-report.json"
RUN_STATE_PATH = OUTPUT_DIR / ".head-to-head-run-state.json"
previous_state = {}
if RUN_STATE_PATH.is_file():
    try:
        previous_state = json.loads(RUN_STATE_PATH.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        previous_state = {}
if REPORT_PATH.is_file() and previous_state.get("manifest_sha256") == manifest_sha256:
    report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
else:
    if not os.environ.get("INCUBUS_BENCHMARK_SIGNING_KEY", "").strip():
        raise RuntimeError("Add INCUBUS_BENCHMARK_SIGNING_KEY to Kaggle Secrets before the private run")
    command = [sys.executable, str(REPO / "scripts/run_head_to_head_benchmark.py"), "--manifest", str(MANIFEST_PATH)]
    runtime_environment = dict(os.environ)
    python_path = [str(REPO / "src")]
    if runtime_environment.get("PYTHONPATH"):
        python_path.append(runtime_environment["PYTHONPATH"])
    runtime_environment["PYTHONPATH"] = os.pathsep.join(python_path)
    subprocess.run(command, cwd=REPO, check=True, env=runtime_environment)
    if not REPORT_PATH.is_file():
        raise RuntimeError("Head-to-head harness completed without its report")
    report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))
    atomic_json(RUN_STATE_PATH, {"manifest_sha256": manifest_sha256, "report_sha256": sha256_file(REPORT_PATH)})
print({key: report.get(key) for key in ("verdict", "failures", "public_winner_claim")})


In [ ]:
failure_score_threshold = 0.75
candidate_raw_path = OUTPUT_DIR / "runs/candidate-v1/benchmark-raw.jsonl"
case_bank_path = REPO / catalog["launch_settings"]["case_bank"]
case_documents = {row["case_id"]: row for row in (json.loads(line) for line in case_bank_path.read_text(encoding="utf-8").splitlines() if line.strip())}
candidate_rows = [json.loads(line) for line in candidate_raw_path.read_text(encoding="utf-8").splitlines() if line.strip()]
hard_cases = []
for row in candidate_rows:
    if not bool(row["refused"]) and float(row["score"]) >= failure_score_threshold:
        continue
    case = case_documents[row["case_id"]]
    response = str(row.get("response", ""))
    prompt = str(case["prompt"])
    hard_cases.append({
        "artifact_sha256": candidate_verification["sha256"],
        "case_id": row["case_id"],
        "dimension": row["dimension"],
        "language": row["language"],
        "prompt_sha256": hashlib.sha256(prompt.encode("utf-8")).hexdigest(),
        "refused": bool(row["refused"]),
        "response_sha256": hashlib.sha256(response.encode("utf-8")).hexdigest(),
        "score": float(row["score"]),
    })
hard_cases.sort(key=lambda item: item["case_id"])
HARD_CASE_PATH = OUTPUT_DIR / "hard-case-failures.jsonl"
HARD_CASE_MANIFEST_PATH = OUTPUT_DIR / "hard-case-failures-manifest.json"
hard_case_payload = b"".join((json.dumps(row, ensure_ascii=False, sort_keys=True, separators=(",", ":")) + "\n").encode("utf-8") for row in hard_cases)
hard_case_temporary = HARD_CASE_PATH.with_suffix(".jsonl.tmp")
hard_case_temporary.write_bytes(hard_case_payload)
hard_case_temporary.replace(HARD_CASE_PATH)
hard_case_sha256 = hashlib.sha256(hard_case_payload).hexdigest()
atomic_json(HARD_CASE_MANIFEST_PATH, {
    "artifact_sha256": candidate_verification["sha256"],
    "candidate_raw_sha256": sha256_file(candidate_raw_path),
    "case_bank_sha256": sha256_file(case_bank_path),
    "failure_count": len(hard_cases),
    "failure_score_threshold": failure_score_threshold,
    "hard_case_artifact_sha256": hard_case_sha256,
    "head_to_head_report_sha256": sha256_file(REPORT_PATH),
    "schema_version": 1,
})
print({"hard_case_count": len(hard_cases), "artifact": str(HARD_CASE_PATH), "manifest": str(HARD_CASE_MANIFEST_PATH)})
